# Load: Population Change and Density (Table 1.1)

Source file: `Table-1.1-Population-population-change-and-population-density.xlsx`  
Source: CBS Aruba and the Population Registry Office

This notebook loads, cleans, reshapes, and persists the data.  
It does **not** produce charts — that lives in `notebooks/eda/`.

In [1]:
import sys
from pathlib import Path
import logging
logging.getLogger("config.project_paths").setLevel(logging.WARNING)

import pandas as pd

sys.path.append(str(Path.cwd().parent.parent if Path.cwd().name == "load" else Path.cwd()))

from config.project_paths import DATA_RAW, DATA_PROCESSED, DB_FILES
from src.data_loader import load_excel_file, save_to_parquet, register_in_duckdb

# Verify
print("DATA_RAW   :", DATA_RAW)
print("PROCESSED  :", DATA_PROCESSED)
print("DB_FILES   :", DB_FILES)

DATA_RAW   : /home/ggirelli/documents/data-analysis/projects/cbs_aruba/data/raw
PROCESSED  : /home/ggirelli/documents/data-analysis/projects/cbs_aruba/data/processed
DB_FILES   : /home/ggirelli/documents/data-analysis/projects/cbs_aruba/outputs/db_files


In [2]:
SOURCE_FILE  = "Table-1.1-Population-population-change-and-population-density.xlsx"
PARQUET_FILE = "population_change_density.parquet"
TABLE_NAME   = "population_change_density"

In [10]:
raw_df = load_excel_file(SOURCE_FILE, sheet_name=0, base_dir=DATA_RAW)

# CBS files have a title row above the header which will be skipped
raw_df = pd.read_excel(DATA_RAW / SOURCE_FILE, header=1)
raw_df = raw_df.dropna(how="all").dropna(axis=1, how="all")
raw_df.columns = [c.strip() if isinstance(c, str) else str(c) for c in raw_df.columns]

print(raw_df.shape)
raw_df.head(10)

INFO:src.data_loader:Loaded Table-1.1-Population-population-change-and-population-density.xlsx from /home/ggirelli/documents/data-analysis/projects/cbs_aruba/data/raw/Table-1.1-Population-population-change-and-population-density.xlsx with shape (8, 10)


(7, 10)


,Unnamed: 0,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Total population1,108635.40,108818.30,108651.30,109164.00,109241.20,107932.00,107468.40,107152.30,107566.30
1,- males,51309.20,51461.60,51398.60,51512.00,51515.10,50930.30,50664.30,50440.20,50559.00
2,- females,57326.20,57356.80,57252.70,57652.00,57726.10,57002.00,56804.10,56712.00,57007.30
4,Annual rate of population change (%)²,1.35,0.17,-0.15,0.47,0.07,-1.21,-0.43,-0.29,0.39
6,Density of population (inhabitants/km²)³,603.50,604.50,603.60,606.50,606.90,599.60,597.00,595.30,597.60
8,Males per 1000 females,895.00,897.20,897.70,893.40,892.40,893.40,891.90,889.40,887.00
9,Source: CBS and Population Registry Office,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
raw_df = raw_df.rename(columns={"Unnamed: 0": "indicator"})

WANTED_ROWS = [
    "Total population1",
    " - males",
    " - females",
    "Annual rate of population change (%)²",
    "Density of population (inhabitants/km²)³",
    "Males per 1000  females",
]

selected_df = raw_df.loc[raw_df["indicator"].isin(WANTED_ROWS)].copy()
print(selected_df.shape)
selected_df

(6, 10)


,indicator,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Total population1,108635.40,108818.30,108651.30,109164.00,109241.20,107932.00,107468.40,107152.30,107566.30
1,- males,51309.20,51461.60,51398.60,51512.00,51515.10,50930.30,50664.30,50440.20,50559.00
2,- females,57326.20,57356.80,57252.70,57652.00,57726.10,57002.00,56804.10,56712.00,57007.30
4,Annual rate of population change (%)²,1.35,0.17,-0.15,0.47,0.07,-1.21,-0.43,-0.29,0.39
6,Density of population (inhabitants/km²)³,603.50,604.50,603.60,606.50,606.90,599.60,597.00,595.30,597.60
8,Males per 1000 females,895.00,897.20,897.70,893.40,892.40,893.40,891.90,889.40,887.00


In [5]:
tidy_df = selected_df.melt(
    id_vars="indicator",
    var_name="year",
    value_name="value",
)
tidy_df["year"] = tidy_df["year"].astype(int)
tidy_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   indicator  54 non-null     str    
 1   year       54 non-null     int64  
 2   value      54 non-null     float64
dtypes: float64(1), int64(1), str(1)
memory usage: 2.6 KB


In [6]:
wide_df = (
    tidy_df
    .pivot(index="year", columns="indicator", values="value")
    .reset_index()
)
wide_df.columns.name = None

wide_df = wide_df.rename(columns={
    " - females":                                    "population_female",
    " - males":                                      "population_male",
    "Annual rate of population change (%)²":         "annual_change_pct",
    "Density of population (inhabitants/km²)³":      "density_per_km2",
    "Males per 1000  females":                       "males_per_1000_females",
    "Total population1":                             "population_total",
})

print(wide_df.dtypes)
wide_df

year                        int64
population_female         float64
population_male           float64
annual_change_pct         float64
density_per_km2           float64
males_per_1000_females    float64
population_total          float64
dtype: object


,year,population_female,population_male,annual_change_pct,density_per_km2,males_per_1000_females,population_total
0,2015,57326.2,51309.2,1.35,603.5,895.0,108635.4
1,2016,57356.8,51461.6,0.17,604.5,897.2,108818.3
2,2017,57252.7,51398.6,-0.15,603.6,897.7,108651.3
3,2018,57652.0,51512.0,0.47,606.5,893.4,109164.0
4,2019,57726.1,51515.1,0.07,606.9,892.4,109241.2
5,2020,57002.0,50930.3,-1.21,599.6,893.4,107932.0
6,2021,56804.1,50664.3,-0.43,597.0,891.9,107468.4
7,2022,56712.0,50440.2,-0.29,595.3,889.4,107152.3
8,2023,57007.3,50559.0,0.39,597.6,887.0,107566.3


In [7]:
wide_df["_check_total"] = wide_df["population_female"] + wide_df["population_male"]
discrepancies = wide_df[abs(wide_df["_check_total"] - wide_df["population_total"]) > 1.0]

if discrepancies.empty:
    print("✓ Male + female sums match population_total within tolerance.")
else:
    print("⚠ Discrepancies found:")
    print(discrepancies[["year", "population_total", "_check_total"]])

wide_df = wide_df.drop(columns=["_check_total"])

✓ Male + female sums match population_total within tolerance.


In [8]:
parquet_path = save_to_parquet(wide_df, PARQUET_FILE, base_dir=DATA_PROCESSED)
register_in_duckdb(parquet_path, TABLE_NAME, db_path=DB_FILES / "aruba.duckdb")

print(f"✓ Parquet saved  : {parquet_path}")
print(f"✓ DuckDB view    : '{TABLE_NAME}' in aruba.duckdb")

INFO:src.data_loader:Saved parquet to /home/ggirelli/documents/data-analysis/projects/cbs_aruba/data/processed/population_change_density.parquet
INFO:src.data_loader:Registered 'population_change_density' in DuckDB at /home/ggirelli/documents/data-analysis/projects/cbs_aruba/outputs/db_files/aruba.duckdb


✓ Parquet saved  : /home/ggirelli/documents/data-analysis/projects/cbs_aruba/data/processed/population_change_density.parquet
✓ DuckDB view    : 'population_change_density' in aruba.duckdb


In [9]:
import duckdb

con = duckdb.connect(str(DB_FILES / "aruba.duckdb"))
result = con.execute(f"SELECT * FROM {TABLE_NAME} ORDER BY year").df()
con.close()

print(result.shape)
result

(9, 7)


,year,population_female,population_male,annual_change_pct,density_per_km2,males_per_1000_females,population_total
0,2015,57326.2,51309.2,1.35,603.5,895.0,108635.4
1,2016,57356.8,51461.6,0.17,604.5,897.2,108818.3
2,2017,57252.7,51398.6,-0.15,603.6,897.7,108651.3
3,2018,57652.0,51512.0,0.47,606.5,893.4,109164.0
4,2019,57726.1,51515.1,0.07,606.9,892.4,109241.2
5,2020,57002.0,50930.3,-1.21,599.6,893.4,107932.0
6,2021,56804.1,50664.3,-0.43,597.0,891.9,107468.4
7,2022,56712.0,50440.2,-0.29,595.3,889.4,107152.3
8,2023,57007.3,50559.0,0.39,597.6,887.0,107566.3
